# Sensitivity of realistic rule gain to the utility table

The economic-rejection finding in `04_rule_categorization.ipynb` depends on
the chosen intrinsic and transition utility tables. This notebook stresses
the categorisation by uniformly rescaling every *negative* (cost-side) entry
in both tables by a factor `s`:

- `cost_x0.5` — intervention costs cut in half.
- `base` — utility tables as published.
- `cost_x1.5` — intervention costs 50 % higher.

Positive entries (notably the target benefit) are left unchanged so that `s`
encodes a *pure cost-to-benefit ratio shift*, not an arbitrary rescaling of
the whole gain.

The uplift CI is invariant to the utility tables so we report it only at
`base`. The realistic-rule-gain CI is recomputed at each scaling.

**Output**: `notebooks/inference_studies/results/utility_table_sensitivity.csv`.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the repository root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

# Import the shared dataset loader (canonical preprocessing for the four
# benchmark datasets used throughout this folder).
sys.path.insert(0, str(REPO_ROOT / 'notebooks' / 'article'))
from _datasets import load_all, load_telco, load_bank_marketing, load_employee_attrition, load_credit_card_default  # noqa: E402

# Outputs for this notebook (kept under the notebook folder so the user can
# inspect them locally without touching the article's canonical CSVs).
RESULTS_DIR = REPO_ROOT / 'notebooks' / 'inference_studies' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root:   {REPO_ROOT}')
print(f'Results dir: {RESULTS_DIR.relative_to(REPO_ROOT)}')


In [ ]:
from collections import Counter
from dataclasses import replace
from typing import Mapping
from action_rules import ActionRules
from _datasets import ALL_LOADERS, DatasetConfig

OUT_CSV = RESULTS_DIR / 'utility_table_sensitivity.csv'

N_BOOTSTRAP = 500
SEED = 42
THRESHOLD = 0.0


In [ ]:
SCALINGS = {
    'cost_x0.5': 0.5,
    'base':      1.0,
    'cost_x1.5': 1.5,
}

def scale_negatives(table, factor: float):
    """Return a shallow copy with every negative value scaled by *factor*."""
    if table is None:
        return None
    return {k: (v * factor if v < 0 else v) for k, v in table.items()}


In [ ]:
def mine_and_categorise(cfg: DatasetConfig) -> dict:
    ar = ActionRules(
        min_stable_attributes=cfg.min_stable_attributes,
        min_flexible_attributes=cfg.min_flexible_attributes,
        min_undesired_support=cfg.min_undesired_support,
        min_desired_support=cfg.min_desired_support,
        min_undesired_confidence=cfg.min_undesired_confidence,
        min_desired_confidence=cfg.min_desired_confidence,
        intrinsic_utility_table=cfg.intrinsic_utility_table,
        transition_utility_table=cfg.transition_utility_table,
    )
    ar.fit(
        cfg.df,
        stable_attributes=cfg.stable_attributes,
        flexible_attributes=cfg.flexible_attributes,
        target=cfg.target,
        target_undesired_state=cfg.target_undesired_state,
        target_desired_state=cfg.target_desired_state,
        use_sparse_matrix=cfg.use_sparse_matrix,
    )
    n_rules = len(ar.output.action_rules) if ar.output is not None else 0
    if n_rules == 0:
        return {
            'n_rules': 0,
            'uplift_accept': 0, 'uplift_uncertain': 0, 'uplift_reject': 0,
            'gain_accept': 0, 'gain_uncertain': 0, 'gain_reject': 0,
        }
    counts = {}
    for metric in ('uplift', 'realistic_rule_gain'):
        results = ar.confidence_intervals(
            cfg.df,
            method='bootstrap',
            confidence_level=0.95,
            threshold=THRESHOLD,
            metric=metric,
            n_bootstrap=N_BOOTSTRAP,
            random_state=SEED,
        )
        counts[metric] = Counter(r.category.value for r in results if r.category is not None)
    return {
        'n_rules': n_rules,
        'uplift_accept':   counts['uplift'].get('accept', 0),
        'uplift_uncertain': counts['uplift'].get('uncertain', 0),
        'uplift_reject':   counts['uplift'].get('reject', 0),
        'gain_accept':     counts['realistic_rule_gain'].get('accept', 0),
        'gain_uncertain':  counts['realistic_rule_gain'].get('uncertain', 0),
        'gain_reject':     counts['realistic_rule_gain'].get('reject', 0),
    }


In [ ]:
rows = []
for ds_name, loader in ALL_LOADERS.items():
    print(f'=== {ds_name} ===')
    base_cfg = loader()
    for cfg_label, factor in SCALINGS.items():
        scaled = replace(
            base_cfg,
            intrinsic_utility_table=scale_negatives(base_cfg.intrinsic_utility_table, factor),
            transition_utility_table=scale_negatives(base_cfg.transition_utility_table, factor),
        )
        print(f'  [{cfg_label}] cost scaling factor={factor}')
        res = mine_and_categorise(scaled)
        res.update({'dataset': ds_name, 'config': cfg_label, 'cost_scaling': factor})
        rows.append(res)
        print(
            f'    -> n={res["n_rules"]}, uplift A/U/R='
            f'{res["uplift_accept"]}/{res["uplift_uncertain"]}/{res["uplift_reject"]},'
            f' gain A/U/R={res["gain_accept"]}/{res["gain_uncertain"]}/{res["gain_reject"]}'
        )

df = pd.DataFrame(rows)[[
    'dataset', 'config', 'cost_scaling',
    'n_rules',
    'uplift_accept', 'uplift_uncertain', 'uplift_reject',
    'gain_accept', 'gain_uncertain', 'gain_reject',
]]
df.to_csv(OUT_CSV, index=False)
print(f'\nwrote {OUT_CSV.relative_to(REPO_ROOT)}')
df


## Reading the sweep

Compare the `gain_*` columns across `cost_x0.5`, `base`, `cost_x1.5`.

- Datasets where the gain categorisation flips substantially under cost
  rescaling (especially Employee Attrition) are economically *fragile*: the
  Accept/Reject verdict depends on the assumed cost structure.
- Datasets where the breakdown stays stable across all three scalings are
  economically *robust*: even a substantial misspecification of the cost side
  doesn't change deployment decisions.

The uplift breakdown is invariant to `s` because the utility tables only
enter the realistic-gain formula; this is a useful sanity check that the
scaling is doing what we think.
